In [7]:
from pathlib import Path
from datetime import datetime
import json
import warnings
import re
import math
import numpy as np
import pandas as pd

from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.section import WD_SECTION
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, KeepTogether
)

warnings.filterwarnings("ignore")

print("=" * 95)
print("RESEARCH PAPER DEVELOPMENT AND EXPORT")
print("=" * 95)

PROJECT_ROOT = Path(
    r"C:\Users\hp\Analysis in Global Supply Chain Operations"
)

CLEANED_FILE = PROJECT_ROOT / "data" / "processed" / "cleaned_logistics_data.csv"
GAP_FILE = PROJECT_ROOT / "data" / "processed" / "delivery_gap_analysis.csv"
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "APL_Logistics.xlsx"

PAPER_DIR = PROJECT_ROOT / "research_paper"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"

PAPER_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

DOCX_FILE = PAPER_DIR / "Global_Supply_Chain_Research_Paper.docx"
PDF_FILE = PAPER_DIR / "Global_Supply_Chain_Research_Paper.pdf"
VERIFICATION_FILE = REPORT_DIR / "research_paper_export_verification.json"

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_ROOT}")

if not CLEANED_FILE.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {CLEANED_FILE}")

if not GAP_FILE.exists():
    raise FileNotFoundError(f"Delivery gap dataset not found: {GAP_FILE}")

print(f"Project directory: {PROJECT_ROOT}")
print(f"Cleaned dataset: {CLEANED_FILE}")
print(f"Delivery gap dataset: {GAP_FILE}")


# LOAD AND VALIDATE THE ACTUAL PROJECT DATA

print("\nLOADING PROJECT DATA")
print("-" * 95)

cleaned_df = pd.read_csv(CLEANED_FILE, low_memory=False)
df = pd.read_csv(GAP_FILE, low_memory=False)

cleaned_df.columns = cleaned_df.columns.astype(str).str.strip()
df.columns = df.columns.astype(str).str.strip()

print(f"Cleaned dataset shape: {cleaned_df.shape}")
print(f"Delivery gap dataset shape: {df.shape}")

def find_column(data, candidates):
    normalized = {
        str(c).strip().lower().replace("_", " ").replace("-", " "): c
        for c in data.columns
    }
    for candidate in candidates:
        key = candidate.strip().lower().replace("_", " ").replace("-", " ")
        if key in normalized:
            return normalized[key]
    return None

actual_col = find_column(
    df, ["Days for shipping (real)", "actual shipping duration"]
)
scheduled_col = find_column(
    df, ["Days for shipment (scheduled)", "scheduled shipping duration"]
)
gap_col = find_column(df, ["Delay Gap", "delivery gap", "delay gap"])
status_col = find_column(df, ["Delivery Status"])
risk_col = find_column(df, ["Late_delivery_risk", "late delivery risk"])
mode_col = find_column(df, ["Shipping Mode"])
region_col = find_column(df, ["Order Region"])
country_col = find_column(df, ["Order Country"])
market_col = find_column(df, ["Market"])
segment_col = find_column(df, ["Customer Segment"])

required_columns = {
    "Actual shipping duration": actual_col,
    "Scheduled shipping duration": scheduled_col,
    "Shipping Mode": mode_col,
    "Order Region": region_col,
    "Market": market_col,
    "Customer Segment": segment_col
}

missing_required = [name for name, col in required_columns.items() if col is None]

if missing_required:
    raise ValueError(
        "Required columns missing from delivery gap dataset: "
        + ", ".join(missing_required)
    )

df[actual_col] = pd.to_numeric(df[actual_col], errors="coerce")
df[scheduled_col] = pd.to_numeric(df[scheduled_col], errors="coerce")

if gap_col:
    df["_gap"] = pd.to_numeric(df[gap_col], errors="coerce")
else:
    df["_gap"] = df[actual_col] - df[scheduled_col]

if risk_col:
    df[risk_col] = pd.to_numeric(df[risk_col], errors="coerce")

cancelled_mask = pd.Series(False, index=df.index)

if status_col:
    cancelled_mask = (
        df[status_col].astype(str)
        .str.strip()
        .str.lower()
        .str.contains("cancel", na=False)
    )

valid_duration = (
    df[actual_col].notna()
    & df[scheduled_col].notna()
    & (df[actual_col] >= 0)
    & (df[scheduled_col] >= 0)
    & df["_gap"].notna()
)

df["_eligible"] = valid_duration & ~cancelled_mask

eligible = df.loc[df["_eligible"]].copy()

if eligible.empty:
    raise ValueError("No eligible shipment records were found.")

eligible["_classification"] = np.select(
    [
        eligible["_gap"] > 0,
        eligible["_gap"] == 0,
        eligible["_gap"] < 0
    ],
    ["Delayed", "Exactly On-time", "Early"],
    default="Invalid"
)

total_records = len(df)
total_cleaned_records = len(cleaned_df)
total_eligible = len(eligible)

print(f"Source analysis records: {total_records:,}")
print(f"Cleaned dataset records: {total_cleaned_records:,}")
print(f"Eligible shipments: {total_eligible:,}")


# CALCULATE OVERALL DELIVERY PERFORMANCE

print("\nCALCULATING DELIVERY PERFORMANCE")
print("-" * 95)

gap = eligible["_gap"]

delayed_count = int((gap > 0).sum())
exact_on_time_count = int((gap == 0).sum())
early_count = int((gap < 0).sum())
sla_compliant_count = int((gap <= 0).sum())

delayed_pct = delayed_count / total_eligible * 100
exact_on_time_pct = exact_on_time_count / total_eligible * 100
early_pct = early_count / total_eligible * 100
sla_compliance_pct = sla_compliant_count / total_eligible * 100

average_delay = (
    float(gap[gap > 0].mean()) if delayed_count else 0.0
)

median_delay = (
    float(gap[gap > 0].median()) if delayed_count else 0.0
)

maximum_delay = float(gap.max())
minimum_gap = float(gap.min())
overall_mean_gap = float(gap.mean())
overall_median_gap = float(gap.median())

risk_ratio = None
risk_valid_count = 0

if risk_col:
    risk_values = eligible[risk_col]
    risk_values = risk_values[risk_values.isin([0, 1])]
    risk_valid_count = len(risk_values)
    if risk_valid_count:
        risk_ratio = float(risk_values.mean() * 100)

status_distribution = pd.DataFrame()

if status_col:
    status_distribution = (
        eligible[status_col]
        .fillna("Unknown")
        .value_counts()
        .rename_axis("Delivery Status")
        .reset_index(name="Shipment Count")
    )
    status_distribution["Percentage"] = (
        status_distribution["Shipment Count"] / total_eligible * 100
    )

classification_table = pd.DataFrame({
    "Delivery Classification": [
        "Delayed",
        "Exactly On-time",
        "Early",
        "SLA-compliant"
    ],
    "Shipment Count": [
        delayed_count,
        exact_on_time_count,
        early_count,
        sla_compliant_count
    ],
    "Percentage": [
        delayed_pct,
        exact_on_time_pct,
        early_pct,
        sla_compliance_pct
    ]
})

print(f"Delayed shipments: {delayed_count:,} ({delayed_pct:.2f}%)")
print(f"Exactly on-time shipments: {exact_on_time_count:,} ({exact_on_time_pct:.2f}%)")
print(f"Early shipments: {early_count:,} ({early_pct:.2f}%)")
print(f"SLA-compliant shipments: {sla_compliant_count:,} ({sla_compliance_pct:.2f}%)")
print(f"Average delay among delayed shipments: {average_delay:.2f} days")


# CALCULATE SHIPPING MODE PERFORMANCE

print("\nCALCULATING SHIPPING MODE PERFORMANCE")
print("-" * 95)

mode_records = []

for mode, group in eligible.groupby(mode_col, dropna=False):
    mode_gap = group["_gap"]
    positive_delays = mode_gap[mode_gap > 0]

    mode_risk = np.nan

    if risk_col:
        valid_risk = group[risk_col]
        valid_risk = valid_risk[valid_risk.isin([0, 1])]
        if len(valid_risk):
            mode_risk = float(valid_risk.mean() * 100)

    mode_records.append({
        "Shipping Mode": str(mode),
        "Shipment Count": len(group),
        "Delayed Shipments": int((mode_gap > 0).sum()),
        "Exactly On-time Shipments": int((mode_gap == 0).sum()),
        "Early Shipments": int((mode_gap < 0).sum()),
        "SLA-compliant Shipments": int((mode_gap <= 0).sum()),
        "Delayed Percentage": float((mode_gap > 0).mean() * 100),
        "Exact On-time Percentage": float((mode_gap == 0).mean() * 100),
        "SLA Compliance Percentage": float((mode_gap <= 0).mean() * 100),
        "Average Delay Days": (
            float(positive_delays.mean()) if len(positive_delays) else 0.0
        ),
        "Late Risk Percentage": mode_risk
    })

mode_table = pd.DataFrame(mode_records)

if not mode_table.empty:
    inverse_risk = 100 - mode_table["Late Risk Percentage"].fillna(
        mode_table["Late Risk Percentage"].mean()
    )

    max_mode_delay = mode_table["Average Delay Days"].max()

    if max_mode_delay > 0:
        delay_score = (
            1 - mode_table["Average Delay Days"] / max_mode_delay
        ) * 100
    else:
        delay_score = pd.Series(100.0, index=mode_table.index)

    mode_table["Shipping Mode Efficiency Index"] = (
        0.50 * mode_table["SLA Compliance Percentage"]
        + 0.30 * inverse_risk
        + 0.20 * delay_score
    ).clip(0, 100).round(2)

    mode_table = mode_table.sort_values(
        "Shipping Mode Efficiency Index",
        ascending=False
    ).reset_index(drop=True)

    mode_table.insert(0, "Efficiency Rank", range(1, len(mode_table) + 1))

best_mode = (
    str(mode_table.iloc[0]["Shipping Mode"])
    if not mode_table.empty else "Not available"
)

worst_mode_delay = (
    str(mode_table.sort_values(
        "Average Delay Days", ascending=False
    ).iloc[0]["Shipping Mode"])
    if not mode_table.empty else "Not available"
)

print(mode_table.to_string(index=False))


# CALCULATE REGIONAL PERFORMANCE

print("\nCALCULATING REGIONAL PERFORMANCE")
print("-" * 95)

region_records = []

for region, group in eligible.groupby(region_col, dropna=False):
    region_gap = group["_gap"]
    positive = region_gap[region_gap > 0]

    region_records.append({
        "Order Region": str(region),
        "Shipment Count": len(group),
        "Delayed Shipments": int((region_gap > 0).sum()),
        "Exactly On-time Shipments": int((region_gap == 0).sum()),
        "Early Shipments": int((region_gap < 0).sum()),
        "SLA-compliant Shipments": int((region_gap <= 0).sum()),
        "Delayed Percentage": float((region_gap > 0).mean() * 100),
        "SLA Compliance Percentage": float((region_gap <= 0).mean() * 100),
        "Average Delay Days": (
            float(positive.mean()) if len(positive) else 0.0
        )
    })

region_table = pd.DataFrame(region_records)

if not region_table.empty:
    max_regional_delay = region_table["Average Delay Days"].max()

    region_table["Regional Delay Index"] = (
        region_table["Average Delay Days"] / max_regional_delay * 100
        if max_regional_delay > 0 else 0.0
    )

    region_table = region_table.sort_values(
        "Delayed Percentage", ascending=False
    ).reset_index(drop=True)

    region_table["Regional Delay Index"] = (
        region_table["Regional Delay Index"].round(2)
    )

highest_delay_region = (
    str(region_table.iloc[0]["Order Region"])
    if not region_table.empty else "Not available"
)

highest_average_delay_region = (
    str(region_table.sort_values(
        "Average Delay Days", ascending=False
    ).iloc[0]["Order Region"])
    if not region_table.empty else "Not available"
)


# CALCULATE MARKET PERFORMANCE

print("\nCALCULATING MARKET PERFORMANCE")
print("-" * 95)

market_records = []

for market, group in eligible.groupby(market_col, dropna=False):
    market_gap = group["_gap"]
    positive = market_gap[market_gap > 0]

    market_records.append({
        "Market": str(market),
        "Shipment Count": len(group),
        "Delayed Shipments": int((market_gap > 0).sum()),
        "Exactly On-time Shipments": int((market_gap == 0).sum()),
        "Early Shipments": int((market_gap < 0).sum()),
        "SLA-compliant Shipments": int((market_gap <= 0).sum()),
        "Delayed Percentage": float((market_gap > 0).mean() * 100),
        "SLA Compliance Percentage": float((market_gap <= 0).mean() * 100),
        "Average Delay Days": (
            float(positive.mean()) if len(positive) else 0.0
        )
    })

market_table = pd.DataFrame(market_records)

if not market_table.empty:
    market_table = market_table.sort_values(
        "Delayed Percentage", ascending=False
    ).reset_index(drop=True)

highest_delay_market = (
    str(market_table.iloc[0]["Market"])
    if not market_table.empty else "Not available"
)


# CALCULATE COUNTRY PERFORMANCE

print("\nCALCULATING COUNTRY PERFORMANCE")
print("-" * 95)

country_table = pd.DataFrame()

if country_col:
    country_records = []

    for country, group in eligible.groupby(country_col, dropna=False):
        country_gap = group["_gap"]
        positive = country_gap[country_gap > 0]

        country_records.append({
            "Order Country": str(country),
            "Shipment Count": len(group),
            "Delayed Shipments": int((country_gap > 0).sum()),
            "Delayed Percentage": float((country_gap > 0).mean() * 100),
            "Average Delay Days": (
                float(positive.mean()) if len(positive) else 0.0
            )
        })

    country_table = pd.DataFrame(country_records)

    if not country_table.empty:
        country_table = country_table.sort_values(
            "Delayed Percentage", ascending=False
        ).reset_index(drop=True)


# CALCULATE CUSTOMER SEGMENT PERFORMANCE

print("\nCALCULATING CUSTOMER SEGMENT PERFORMANCE")
print("-" * 95)

segment_records = []

for segment, group in eligible.groupby(segment_col, dropna=False):
    segment_gap = group["_gap"]
    positive = segment_gap[segment_gap > 0]

    segment_risk = np.nan

    if risk_col:
        valid_risk = group[risk_col]
        valid_risk = valid_risk[valid_risk.isin([0, 1])]
        if len(valid_risk):
            segment_risk = float(valid_risk.mean() * 100)

    segment_records.append({
        "Customer Segment": str(segment),
        "Shipment Count": len(group),
        "Delayed Shipments": int((segment_gap > 0).sum()),
        "Exactly On-time Shipments": int((segment_gap == 0).sum()),
        "Early Shipments": int((segment_gap < 0).sum()),
        "SLA-compliant Shipments": int((segment_gap <= 0).sum()),
        "Delayed Percentage": float((segment_gap > 0).mean() * 100),
        "Exact On-time Percentage": float((segment_gap == 0).mean() * 100),
        "SLA Compliance Percentage": float((segment_gap <= 0).mean() * 100),
        "Average Delay Days": (
            float(positive.mean()) if len(positive) else 0.0
        ),
        "Late Risk Percentage": segment_risk
    })

segment_table = pd.DataFrame(segment_records)

if not segment_table.empty:
    segment_table = segment_table.sort_values(
        "Delayed Percentage", ascending=False
    ).reset_index(drop=True)

highest_delay_segment = (
    str(segment_table.iloc[0]["Customer Segment"])
    if not segment_table.empty else "Not available"
)


# CALCULATE THE FIVE REQUIRED PROJECT KPIs

print("\nCALCULATING THE FIVE REQUIRED KPIs")
print("-" * 95)

on_time_delivery_rate = sla_compliance_pct

average_delivery_delay = average_delay

late_delivery_risk_ratio = (
    risk_ratio if risk_ratio is not None else np.nan
)

shipping_mode_efficiency_index = (
    float(mode_table["Shipping Mode Efficiency Index"].max())
    if not mode_table.empty else np.nan
)

regional_delay_index = (
    float(region_table["Regional Delay Index"].max())
    if not region_table.empty else np.nan
)

kpi_table = pd.DataFrame([
    {
        "KPI": "On-Time Delivery Rate (%)",
        "Value": on_time_delivery_rate,
        "Calculation": "SLA-compliant shipments / eligible shipments × 100"
    },
    {
        "KPI": "Average Delivery Delay (Days)",
        "Value": average_delivery_delay,
        "Calculation": "Mean positive delay gap among delayed shipments"
    },
    {
        "KPI": "Late Delivery Risk Ratio (%)",
        "Value": late_delivery_risk_ratio,
        "Calculation": "Mean valid binary Late_delivery_risk label × 100"
    },
    {
        "KPI": "Shipping Mode Efficiency Index",
        "Value": shipping_mode_efficiency_index,
        "Calculation": "50% SLA compliance + 30% inverse risk + 20% delay score"
    },
    {
        "KPI": "Regional Delay Index",
        "Value": regional_delay_index,
        "Calculation": "Regional average positive delay / maximum regional average positive delay × 100"
    }
])

print(kpi_table.to_string(index=False))


# CHECK DATASET LIMITATIONS AND ACTUAL DATA CHARACTERISTICS

print("\nDATASET CHARACTERISTICS AND LIMITATIONS")
print("-" * 95)

date_candidates = [
    "date", "order date", "shipping date", "delivery date"
]

date_columns = [
    c for c in df.columns
    if str(c).strip().lower().replace("_", " ") in date_candidates
]

has_genuine_date_column = len(date_columns) > 0

zero_scheduled_count = int((df[scheduled_col] == 0).sum())
cancelled_count = int(cancelled_mask.sum())
invalid_duration_count = int((~valid_duration).sum())

duplicate_count = int(df.duplicated().sum())

source_column_count = len(cleaned_df.columns)
source_row_count = len(cleaned_df)

segment_categories = sorted(
    eligible[segment_col].dropna().astype(str).unique().tolist()
)

market_categories = sorted(
    eligible[market_col].dropna().astype(str).unique().tolist()
)

region_categories = sorted(
    eligible[region_col].dropna().astype(str).unique().tolist()
)

mode_categories = sorted(
    eligible[mode_col].dropna().astype(str).unique().tolist()
)

limitations = [
    "The processed dataset does not contain a verified operational date column; chronological trend and seasonality analysis cannot be established.",
    "No synthetic dates have been created.",
    "The dataset does not provide a dedicated Order ID for order-level deduplication.",
    "Customer coordinates are not treated as confirmed shipment destination coordinates.",
    "Late_delivery_risk is an existing binary dataset label, not a newly trained probability model.",
    "Customer segments are interpreted as Consumer, Corporate and Home Office; Corporate is not assumed to mean premium or enterprise.",
    "The analysis is based on shipment records and available fields; causal explanations cannot be established from these descriptive associations alone.",
    "The Shipping Mode Efficiency Index and Regional Delay Index are project-defined analytical measures, not externally standardized industry benchmarks."
]

print(f"Date column available: {has_genuine_date_column}")
print(f"Zero scheduled-duration records: {zero_scheduled_count:,}")
print(f"Cancelled records: {cancelled_count:,}")
print(f"Records with invalid duration fields: {invalid_duration_count:,}")
print(f"Exact duplicate rows in gap dataset: {duplicate_count:,}")
print(f"Regions: {len(region_categories)}")
print(f"Markets: {len(market_categories)}")
print(f"Countries: {len(country_table)}")
print(f"Shipping modes: {len(mode_categories)}")
print(f"Customer segments: {len(segment_categories)}")


# PREPARE ACTUAL FINDINGS FOR THE PAPER

print("\nPREPARING RESEARCH FINDINGS")
print("-" * 95)

largest_mode = (
    str(mode_table.sort_values(
        "Shipment Count", ascending=False
    ).iloc[0]["Shipping Mode"])
    if not mode_table.empty else "Not available"
)

largest_market = (
    str(market_table.sort_values(
        "Shipment Count", ascending=False
    ).iloc[0]["Market"])
    if not market_table.empty else "Not available"
)

largest_segment = (
    str(segment_table.sort_values(
        "Shipment Count", ascending=False
    ).iloc[0]["Customer Segment"])
    if not segment_table.empty else "Not available"
)

largest_region = (
    str(region_table.sort_values(
        "Shipment Count", ascending=False
    ).iloc[0]["Order Region"])
    if not region_table.empty else "Not available"
)

findings = [
    f"The analysis contains {total_eligible:,} eligible shipment records after excluding cancelled shipments and records with invalid or missing duration values.",
    f"{delayed_count:,} eligible shipments ({delayed_pct:.2f}%) have a positive delivery gap and are classified as delayed.",
    f"{exact_on_time_count:,} shipments ({exact_on_time_pct:.2f}%) have a zero delivery gap and are exactly on-time.",
    f"{early_count:,} shipments ({early_pct:.2f}%) have a negative delivery gap and are classified as early.",
    f"{sla_compliant_count:,} shipments ({sla_compliance_pct:.2f}%) are SLA-compliant under the project's definition of a gap less than or equal to zero.",
    f"The average delay among delayed shipments is {average_delay:.2f} days, while the median positive delay is {median_delay:.2f} days.",
    f"The existing Late_delivery_risk label has a ratio of {risk_ratio:.2f}% among valid binary risk records." if risk_ratio is not None else "The late delivery risk ratio could not be calculated because a valid risk field was unavailable.",
    f"{largest_mode} has the largest shipment volume among shipping modes.",
    f"{best_mode} has the highest project-defined Shipping Mode Efficiency Index.",
    f"{worst_mode_delay} has the highest average positive delay among shipping modes.",
    f"{largest_region} has the largest shipment volume among regions.",
    f"{highest_average_delay_region} has the highest regional average positive delay.",
    f"{largest_market} has the largest shipment volume among markets.",
    f"{highest_delay_market} has the highest delayed-shipment percentage among markets.",
    f"{largest_segment} has the largest shipment volume among customer segments.",
    f"{highest_delay_segment} has the highest delayed-shipment percentage among customer segments."
]

if country_col and not country_table.empty:
    top_country = str(country_table.iloc[0]["Order Country"])
    findings.append(
        f"{top_country} has the highest delayed-shipment percentage in the country-level comparison."
    )

bottlenecks = [
    f"Overall delayed-shipment exposure is {delayed_pct:.2f}% of eligible shipments.",
    f"The shipping mode with the highest average positive delay is {worst_mode_delay}.",
    f"The region with the highest average positive delay is {highest_average_delay_region}.",
    f"The market with the highest delayed-shipment percentage is {highest_delay_market}.",
    f"The customer segment with the highest delayed-shipment percentage is {highest_delay_segment}.",
    "The absence of a genuine operational date column restricts chronological monitoring, trend analysis and seasonality assessment."
]

recommendations = [
    "Prioritize operational review of the shipping mode with the highest average positive delay and compare its shipment volume, SLA compliance and existing risk-label distribution.",
    "Investigate the regions and markets with the highest observed delay measures to identify operational causes using additional verified logistics records.",
    "Use the exact on-time, early and delayed classifications separately in performance reporting so that early deliveries are not confused with exactly on-time deliveries.",
    "Monitor SLA compliance and average delay together. A single percentage does not describe the magnitude of delays among shipments that miss their scheduled duration.",
    "Review the existing Late_delivery_risk label against observed delivery outcomes before using it as an operational decision signal.",
    "Assess customer segments using their observed shipment counts, delay percentages and SLA compliance without assigning unsupported premium or enterprise categories.",
    "Collect and validate genuine order, dispatch and delivery timestamps in future data releases to enable chronological forecasting, trend analysis and date-based operational monitoring.",
    "Maintain consistent shipment-level data quality checks, including validation of actual and scheduled durations and clear treatment of cancelled shipments."
]


# SAVE ANALYTICAL TABLES USED IN THE PAPER

print("\nSAVING RESEARCH TABLES")
print("-" * 95)

tables_to_save = {
    "research_delivery_classification.csv": classification_table,
    "research_shipping_mode_performance.csv": mode_table,
    "research_regional_performance.csv": region_table,
    "research_market_performance.csv": market_table,
    "research_country_performance.csv": country_table,
    "research_customer_segment_performance.csv": segment_table,
    "research_required_kpis.csv": kpi_table,
    "research_delivery_status_distribution.csv": status_distribution
}

for filename, table in tables_to_save.items():
    if table is not None and not table.empty:
        table.to_csv(TABLE_DIR / filename, index=False)
        print(f"Saved: {filename}")


# BUILD THE COMPLETE RESEARCH PAPER CONTENT

title = (
    "Delivery Performance, Delay Risk, and Logistics Efficiency "
    "Analysis in Global Supply Chain Operations"
)

abstract = (
    f"This research examines delivery performance, delay exposure and "
    f"logistics efficiency using the project's processed supply-chain "
    f"shipment dataset. The analysis uses {total_eligible:,} eligible "
    f"shipment records after excluding cancelled shipments and records "
    f"with invalid or missing actual or scheduled shipping durations. "
    f"Delivery gaps are calculated as actual shipping duration minus "
    f"scheduled shipping duration. The observed results show that "
    f"{delayed_pct:.2f}% of eligible shipments were delayed, "
    f"{exact_on_time_pct:.2f}% were exactly on-time, and "
    f"{early_pct:.2f}% were early. Under the project's SLA-based KPI "
    f"definition, the On-Time Delivery Rate is {sla_compliance_pct:.2f}%. "
    f"The average delay among delayed shipments is {average_delay:.2f} days. "
    f"The study compares shipping modes, regions, countries, markets and "
    f"customer segments, and applies five project-defined logistics KPIs. "
    f"The findings are descriptive and reflect the fields available in "
    f"the dataset. The absence of a verified operational date field limits "
    f"chronological and seasonality analysis. The paper concludes with "
    f"evidence-based operational recommendations grounded in the observed "
    f"shipment records."
)

introduction = (
    "Global supply chains depend on the coordination of transportation, "
    "shipment scheduling, regional distribution and customer fulfilment. "
    "Delivery performance is an important operational dimension because "
    "departures from scheduled shipping durations can affect service "
    "reliability, logistics planning and the allocation of operational "
    "resources. A structured analysis of actual and scheduled shipping "
    "durations can identify the extent of delivery delays and reveal "
    "differences across shipping modes, markets, regions and customer "
    "segments.\n\n"
    "This project examines shipment-level delivery performance using "
    "the supplied APL Logistics dataset. It calculates delivery gaps, "
    "classifies shipments as delayed, exactly on-time or early, and "
    "develops five required logistics KPIs. The analysis is descriptive: "
    "it identifies patterns in the available records without claiming "
    "that observed associations establish causal relationships."
)

background = (
    "Logistics performance measurement commonly distinguishes the "
    "scheduled duration of a shipment from the duration actually recorded. "
    "The difference provides a direct measure of whether a shipment took "
    "longer than planned, matched the schedule or arrived earlier than "
    "the scheduled duration. These classifications support comparisons "
    "between shipping modes and geographic groupings.\n\n"
    "In this project, SLA compliance is defined as a delivery gap less "
    "than or equal to zero. This includes both exactly on-time and early "
    "shipments. Exact on-time delivery is separately defined as a gap "
    "equal to zero. This distinction is necessary because an SLA-compliant "
    "percentage should not be interpreted as the percentage of shipments "
    "with an exact zero-day gap."
)

problem_statement = (
    "The project addresses the need for a consistent, evidence-based "
    "assessment of delivery performance across a large set of logistics "
    "records. Without a common delivery-gap calculation and comparable "
    "performance indicators, it is difficult to distinguish delayed "
    "shipments from early and exactly on-time shipments or compare "
    "operational performance across shipping modes, regions, markets "
    "and customer segments.\n\n"
    "The analysis also needs to account for cancelled shipments and "
    "invalid duration records, preserve valid zero-duration Same Day "
    "shipments, and avoid unsupported chronological or geographic "
    "interpretations."
)

objectives = [
    "Assess the quality and structure of the processed logistics dataset.",
    "Calculate delivery gaps using actual and scheduled shipping durations.",
    "Classify eligible shipments as delayed, exactly on-time or early.",
    "Measure overall delivery performance and the existing late delivery risk label.",
    "Compare delivery performance and efficiency across shipping modes.",
    "Evaluate regional, country and market-level delivery differences.",
    "Assess delivery performance across Consumer, Corporate and Home Office segments.",
    "Calculate the five required project logistics KPIs.",
    "Identify observed operational bottlenecks and formulate evidence-based recommendations."
]

dataset_description = (
    f"The cleaned dataset contains {source_row_count:,} records and "
    f"{source_column_count} columns. The delivery-gap analysis dataset "
    f"contains {total_records:,} records and {len(df.columns)} columns. "
    f"The cleaned dataset is stored at "
    f"`data/processed/cleaned_logistics_data.csv`, while the delivery-gap "
    f"analysis dataset is stored at "
    f"`data/processed/delivery_gap_analysis.csv`.\n\n"
    f"The analytical fields used include actual shipping duration, "
    f"scheduled shipping duration, delivery status, late delivery risk, "
    f"shipping mode, order region, order country, market and customer "
    f"segment. The primary unit of analysis is the available shipment "
    f"record. The processed dataset covers {len(region_categories)} "
    f"regions, {len(market_categories)} markets, {len(country_table)} "
    f"countries, {len(mode_categories)} shipping modes and "
    f"{len(segment_categories)} customer segments."
)

dataset_limitations = "\n".join(limitations)

cleaning_methodology = (
    f"The research uses the project's processed logistics datasets. "
    f"The cleaned dataset contains {total_cleaned_records:,} records. "
    f"For delivery performance calculations, actual and scheduled "
    f"durations are converted to numeric values. Records are considered "
    f"eligible when both duration values are present, non-negative and "
    f"the delivery gap is valid. Cancelled shipments are excluded from "
    f"the eligible performance population.\n\n"
    f"Valid zero scheduled-duration records are retained. A zero "
    f"scheduled duration is not automatically treated as an error because "
    f"the dataset contains Same Day shipments. Missing or invalid "
    f"durations are not converted to zero. Personal customer identifiers "
    f"were excluded from the cleaned analytical dataset. The analysis "
    f"does not deduplicate customer IDs as if they were order IDs."
)

gap_methodology = (
    "The delivery gap is calculated using the following formula:\n\n"
    "Delivery Gap = Actual Shipping Duration - Scheduled Shipping Duration\n\n"
    "A positive value indicates that actual shipping took longer than "
    "scheduled and the shipment is classified as delayed. A value of "
    "zero indicates an exactly on-time shipment. A negative value "
    "indicates an early shipment.\n\n"
    "For the project's On-Time Delivery Rate KPI, SLA compliance is "
    "calculated as the number of eligible shipments with a delivery "
    "gap less than or equal to zero divided by all eligible shipments, "
    "multiplied by 100. This project-defined KPI includes early shipments. "
    "Exact on-time performance is separately calculated using a delivery "
    "gap equal to zero."
)

delivery_analysis = (
    f"The eligible population contains {total_eligible:,} shipments. "
    f"Among these records, {delayed_count:,} shipments were delayed, "
    f"{exact_on_time_count:,} were exactly on-time and {early_count:,} "
    f"were early. These categories reconcile to the eligible shipment "
    f"population. The SLA-compliant population contains "
    f"{sla_compliant_count:,} shipments because both exactly on-time "
    f"and early shipments satisfy the project's gap less than or equal "
    f"to zero definition.\n\n"
    f"The delayed-shipment percentage is {delayed_pct:.2f}%. The exact "
    f"on-time percentage is {exact_on_time_pct:.2f}%, while the early "
    f"shipment percentage is {early_pct:.2f}%. The project's SLA-based "
    f"On-Time Delivery Rate KPI is {sla_compliance_pct:.2f}%. The mean "
    f"positive delivery gap among delayed shipments is "
    f"{average_delay:.2f} days and the median positive delay is "
    f"{median_delay:.2f} days."
)

shipping_analysis = (
    f"Shipping-mode analysis compares shipment volume, delayed shipment "
    f"percentage, exact on-time percentage, SLA compliance, average "
    f"positive delay and the project-defined Shipping Mode Efficiency "
    f"Index. The highest-ranked mode by the calculated efficiency index "
    f"is {best_mode}. The largest shipment volume is associated with "
    f"{largest_mode}. The highest average positive delay is observed "
    f"for {worst_mode_delay}.\n\n"
    f"The efficiency index combines three scaled components: 50% SLA "
    f"compliance, 30% inverse late delivery risk and 20% relative "
    f"delay-gap score. The index is intended for comparison within this "
    f"project and should not be interpreted as an externally standardized "
    f"industry benchmark."
)

regional_analysis = (
    f"Regional analysis compares shipment volume, delayed-shipment "
    f"percentage, SLA compliance and average positive delivery delay. "
    f"{largest_region} has the largest shipment volume in the regional "
    f"comparison. {highest_average_delay_region} has the highest average "
    f"positive delay. The Regional Delay Index scales each region's "
    f"average positive delay against the maximum regional average "
    f"positive delay, with the maximum set to 100.\n\n"
    f"Market-level analysis identifies {largest_market} as the largest "
    f"market by shipment volume and {highest_delay_market} as the market "
    f"with the highest delayed-shipment percentage. Country-level "
    f"comparisons are descriptive and depend on the number of records "
    f"available for each country. These comparisons do not establish "
    f"why a particular geography experiences a higher delay percentage."
)

segment_analysis = (
    f"The customer-segment analysis compares Consumer, Corporate and "
    f"Home Office records using shipment volume, delayed-shipment "
    f"percentage, exact on-time percentage, SLA compliance, average "
    f"positive delay and the available late delivery risk label.\n\n"
    f"{largest_segment} has the largest shipment volume, while "
    f"{highest_delay_segment} has the highest delayed-shipment percentage "
    f"in the segment comparison. Segment labels are interpreted exactly "
    f"as provided in the dataset. Corporate is not treated as a premium "
    f"or enterprise classification."
)

kpi_methodology = (
    "The five required KPIs are calculated as follows:\n\n"
    "On-Time Delivery Rate (%) = SLA-compliant shipments / eligible shipments x 100. "
    "SLA-compliant shipments have a delivery gap less than or equal to zero.\n\n"
    "Average Delivery Delay (Days) = Mean of positive delivery gaps among "
    "delayed eligible shipments only.\n\n"
    "Late Delivery Risk Ratio (%) = Mean of valid binary Late_delivery_risk "
    "labels multiplied by 100.\n\n"
    "Shipping Mode Efficiency Index = 50% SLA compliance score + 30% "
    "inverse late-risk score + 20% relative delay-gap score. The three "
    "components are expressed on a 0-100 scale.\n\n"
    "Regional Delay Index = Regional average positive delay / Maximum "
    "regional average positive delay x 100. The region with the maximum "
    "average positive delay receives an index of 100.\n\n"
    "These definitions are the project's specified KPI formulas. The "
    "Shipping Mode Efficiency Index and Regional Delay Index are "
    "project-defined comparative indicators."
)

results_findings = "\n".join(findings)

operational_bottlenecks = "\n".join(bottlenecks)

business_recommendations = "\n".join(recommendations)

conclusion = (
    f"The project provides a structured descriptive assessment of "
    f"delivery performance using {total_eligible:,} eligible shipment "
    f"records. The delivery-gap method distinguishes delayed, exactly "
    f"on-time and early shipments, while the project-defined SLA "
    f"compliance KPI reports a value of {sla_compliance_pct:.2f}%. "
    f"The average positive delay among delayed shipments is "
    f"{average_delay:.2f} days.\n\n"
    f"Shipping-mode, regional, market and customer-segment comparisons "
    f"identify differences in shipment volume and observed delivery "
    f"performance. The five required KPIs provide a consistent framework "
    f"for comparing operational outcomes. The findings support targeted "
    f"review of the modes and groups with higher observed delay measures.\n\n"
    f"The results must be interpreted within the limitations of the "
    f"available data. In particular, the absence of a genuine operational "
    f"date field prevents chronological trend and seasonality analysis. "
    f"Further work would benefit from validated order and delivery "
    f"timestamps and additional operational variables that can help "
    f"investigate the causes of delay."
)

references = [
    "APL Logistics dataset supplied for the project, original workbook: data/raw/APL_Logistics.xlsx.",
    "Project processed dataset: data/processed/cleaned_logistics_data.csv.",
    "Project delivery-gap analysis dataset: data/processed/delivery_gap_analysis.csv.",
    "Project analysis outputs: shipping mode, regional, market, customer-segment and KPI analysis tables generated from the supplied dataset.",
    "Pandas documentation. https://pandas.pydata.org/docs/.",
    "NumPy documentation. https://numpy.org/doc/.",
    "Plotly Python documentation. https://plotly.com/python/.",
    "Streamlit documentation. https://docs.streamlit.io/."
]


# DEFINE TABLES FOR THE RESEARCH PAPER

paper_tables = [
    ("Delivery Classification Results", classification_table),
    ("Required Logistics KPIs", kpi_table),
    ("Shipping Mode Performance", mode_table),
    ("Regional Performance", region_table),
    ("Market Performance", market_table),
    ("Customer Segment Performance", segment_table),
]

if not country_table.empty:
    paper_tables.insert(5, ("Country Performance - Highest Delay Percentages", country_table.head(15)))


# CREATE THE WORD DOCUMENT

print("\nCREATING RESEARCH PAPER DOCX")
print("-" * 95)

doc = Document()

section = doc.sections[0]
section.top_margin = Inches(0.7)
section.bottom_margin = Inches(0.7)
section.left_margin = Inches(0.75)
section.right_margin = Inches(0.75)

styles = doc.styles
styles["Normal"].font.name = "Times New Roman"
styles["Normal"].font.size = Pt(11)
styles["Normal"].paragraph_format.space_after = Pt(7)

for style_name in ["Title", "Heading 1", "Heading 2"]:
    styles[style_name].font.name = "Times New Roman"

styles["Heading 1"].font.size = Pt(15)
styles["Heading 1"].font.bold = True
styles["Heading 1"].font.color.rgb = RGBColor(31, 78, 121)

styles["Heading 2"].font.size = Pt(12)
styles["Heading 2"].font.bold = True

# Header and footer

header = section.header.paragraphs[0]
header.text = "GLOBAL SUPPLY CHAIN OPERATIONS | RESEARCH PAPER"
header.alignment = WD_ALIGN_PARAGRAPH.CENTER
header.runs[0].font.size = Pt(8)
header.runs[0].font.color.rgb = RGBColor(100, 100, 100)

footer = section.footer.paragraphs[0]
footer.alignment = WD_ALIGN_PARAGRAPH.CENTER
footer_run = footer.add_run("Research Paper | Global Supply Chain Operations")
footer_run.font.size = Pt(8)

# Title page

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
p.paragraph_format.space_before = Pt(90)

run = p.add_run(title)
run.bold = True
run.font.name = "Times New Roman"
run.font.size = Pt(23)
run.font.color.rgb = RGBColor(31, 78, 121)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
p.paragraph_format.space_before = Pt(25)
r = p.add_run("Research Paper")
r.bold = True
r.font.size = Pt(16)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
p.paragraph_format.space_before = Pt(20)
p.add_run(
    "Project: Analysis in Global Supply Chain Operations\n"
    "Research focus: Delivery Performance, Delay Risk and Logistics Efficiency\n\n"
    f"Dataset records analyzed: {total_eligible:,}\n"
    f"Generated: {datetime.now().strftime('%d %B %Y')}"
)

doc.add_page_break()


def add_doc_paragraph(text, justify=True):
    for block in str(text).split("\n\n"):
        if not block.strip():
            continue
        p = doc.add_paragraph()
        if justify:
            p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
        p.add_run(block.strip())


def shade_cell(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)


def add_doc_table(dataframe, title_text, max_rows=30):
    if dataframe is None or dataframe.empty:
        return

    doc.add_heading(title_text, level=2)

    view = dataframe.head(max_rows).copy()
    view = view.replace({np.nan: ""})

    table = doc.add_table(rows=1, cols=len(view.columns))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.style = "Table Grid"

    for i, column in enumerate(view.columns):
        cell = table.rows[0].cells[i]
        cell.text = str(column)
        shade_cell(cell, "1F4E78")
        for paragraph in cell.paragraphs:
            for run in paragraph.runs:
                run.bold = True
                run.font.color.rgb = RGBColor(255, 255, 255)
                run.font.size = Pt(8)

    for _, row in view.iterrows():
        cells = table.add_row().cells
        for i, value in enumerate(row):
            if isinstance(value, (float, np.floating)):
                text = f"{value:,.2f}"
            elif isinstance(value, (int, np.integer)):
                text = f"{value:,}"
            else:
                text = str(value)

            cells[i].text = text
            cells[i].vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

            for paragraph in cells[i].paragraphs:
                for run in paragraph.runs:
                    run.font.size = Pt(8)

    doc.add_paragraph()


# Paper sections

doc.add_heading("Abstract", level=1)
add_doc_paragraph(abstract)

doc.add_heading("Introduction", level=1)
add_doc_paragraph(introduction)

doc.add_heading("Background and Context", level=1)
add_doc_paragraph(background)

doc.add_heading("Problem Statement", level=1)
add_doc_paragraph(problem_statement)

doc.add_heading("Project Objectives", level=1)
for item in objectives:
    doc.add_paragraph(item, style="List Bullet")

doc.add_heading("Dataset Description", level=1)
add_doc_paragraph(dataset_description)

dataset_summary_table = pd.DataFrame([
    {"Dataset Characteristic": "Cleaned dataset records", "Value": total_cleaned_records},
    {"Dataset Characteristic": "Cleaned dataset columns", "Value": source_column_count},
    {"Dataset Characteristic": "Delivery gap dataset records", "Value": total_records},
    {"Dataset Characteristic": "Eligible shipments", "Value": total_eligible},
    {"Dataset Characteristic": "Regions", "Value": len(region_categories)},
    {"Dataset Characteristic": "Markets", "Value": len(market_categories)},
    {"Dataset Characteristic": "Countries", "Value": len(country_table)},
    {"Dataset Characteristic": "Shipping modes", "Value": len(mode_categories)},
    {"Dataset Characteristic": "Customer segments", "Value": len(segment_categories)},
    {"Dataset Characteristic": "Verified operational date column", "Value": "Available" if has_genuine_date_column else "Not available"}
])
add_doc_table(dataset_summary_table, "Dataset Summary")

doc.add_heading("Dataset Limitations", level=1)
for item in limitations:
    doc.add_paragraph(item, style="List Bullet")

doc.add_heading("Data Cleaning Methodology", level=1)
add_doc_paragraph(cleaning_methodology)

doc.add_heading("Delivery Gap Calculation Methodology", level=1)
add_doc_paragraph(gap_methodology)

doc.add_heading("Delivery Performance Analysis", level=1)
add_doc_paragraph(delivery_analysis)
add_doc_table(classification_table, "Delivery Classification Results")

doc.add_heading("Shipping Mode Efficiency Analysis", level=1)
add_doc_paragraph(shipping_analysis)
add_doc_table(mode_table, "Shipping Mode Performance")

doc.add_heading("Regional and Market Analysis", level=1)
add_doc_paragraph(regional_analysis)
add_doc_table(region_table, "Regional Performance")
add_doc_table(market_table, "Market Performance")
if not country_table.empty:
    add_doc_table(country_table.head(15), "Country Performance - Highest Delay Percentages")

doc.add_heading("Customer Segment Impact Analysis", level=1)
add_doc_paragraph(segment_analysis)
add_doc_table(segment_table, "Customer Segment Performance")

doc.add_heading("KPI Methodology", level=1)
add_doc_paragraph(kpi_methodology)
add_doc_table(kpi_table, "Five Required Logistics KPIs")

doc.add_heading("Results and Findings", level=1)
for item in findings:
    doc.add_paragraph(item, style="List Bullet")

doc.add_heading("Operational Bottlenecks", level=1)
for item in bottlenecks:
    doc.add_paragraph(item, style="List Bullet")

doc.add_heading("Business Recommendations", level=1)
for item in recommendations:
    doc.add_paragraph(item, style="List Number")

doc.add_heading("Conclusion", level=1)
add_doc_paragraph(conclusion)

doc.add_heading("References", level=1)
for item in references:
    doc.add_paragraph(item, style="List Number")

doc.save(DOCX_FILE)

print(f"DOCX created: {DOCX_FILE}")
print(f"DOCX size: {DOCX_FILE.stat().st_size:,} bytes")


# CREATE THE PDF DOCUMENT

print("\nCREATING RESEARCH PAPER PDF")
print("-" * 95)

pdf_styles = getSampleStyleSheet()

pdf_styles.add(ParagraphStyle(
    name="PaperTitleCustom",
    parent=pdf_styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=21,
    leading=27,
    alignment=TA_CENTER,
    textColor=colors.HexColor("#1F4E78"),
    spaceAfter=22
))

pdf_styles.add(ParagraphStyle(
    name="PaperHeading",
    parent=pdf_styles["Heading1"],
    fontName="Helvetica-Bold",
    fontSize=14,
    leading=18,
    textColor=colors.HexColor("#1F4E78"),
    spaceBefore=13,
    spaceAfter=8
))

pdf_styles.add(ParagraphStyle(
    name="PaperSubHeading",
    parent=pdf_styles["Heading2"],
    fontName="Helvetica-Bold",
    fontSize=10.5,
    leading=14,
    spaceBefore=9,
    spaceAfter=6
))

pdf_styles.add(ParagraphStyle(
    name="PaperBody",
    parent=pdf_styles["BodyText"],
    fontName="Helvetica",
    fontSize=9,
    leading=13,
    alignment=TA_JUSTIFY,
    spaceAfter=8
))

pdf_styles.add(ParagraphStyle(
    name="PaperBullet",
    parent=pdf_styles["BodyText"],
    fontName="Helvetica",
    fontSize=8.8,
    leading=12,
    leftIndent=15,
    firstLineIndent=-8,
    spaceAfter=5
))

pdf_styles.add(ParagraphStyle(
    name="PaperSmall",
    parent=pdf_styles["BodyText"],
    fontName="Helvetica",
    fontSize=7,
    leading=9
))


def pdf_safe(value):
    text = str(value)
    text = text.replace("≤", "<=")
    text = text.replace("≥", ">=")
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    text = text.replace("×", "x")
    text = text.replace("−", "-")
    return text


def pdf_paragraph(text, style="PaperBody"):
    safe = pdf_safe(text)
    safe = safe.replace("&", "&amp;")
    safe = safe.replace("<=", "&lt;=")
    safe = safe.replace(">=", "&gt;=")
    return Paragraph(safe.replace("\n", "<br/>"), pdf_styles[style])


def pdf_add_text(story, text):
    for block in str(text).split("\n\n"):
        if block.strip():
            story.append(pdf_paragraph(block.strip()))
            story.append(Spacer(1, 3))


def pdf_add_table(story, dataframe, title_text, max_rows=30):
    if dataframe is None or dataframe.empty:
        return

    story.append(pdf_paragraph(title_text, "PaperSubHeading"))

    view = dataframe.head(max_rows).copy()
    headers = [pdf_safe(c) for c in view.columns]
    rows = [headers]

    for _, row in view.iterrows():
        output_row = []
        for value in row:
            if pd.isna(value):
                output_row.append("")
            elif isinstance(value, (float, np.floating)):
                output_row.append(f"{value:,.2f}")
            elif isinstance(value, (int, np.integer)):
                output_row.append(f"{value:,}")
            else:
                output_row.append(pdf_safe(value))
        rows.append(output_row)

    usable_width = A4[0] - 1.15 * inch
    col_width = usable_width / max(1, len(headers))

    table = Table(
        rows,
        repeatRows=1,
        colWidths=[col_width] * len(headers)
    )

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F4E78")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 6.5),
        ("LEADING", (0, 0), (-1, -1), 8),
        ("GRID", (0, 0), (-1, -1), 0.35, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [
            colors.white, colors.HexColor("#EEF3F8")
        ]),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4)
    ]))

    story.append(table)
    story.append(Spacer(1, 10))


def add_page_number(canvas, doc_template):
    canvas.saveState()
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(colors.grey)
    canvas.drawCentredString(
        A4[0] / 2,
        0.35 * inch,
        f"Global Supply Chain Operations | Page {doc_template.page}"
    )
    canvas.restoreState()


story = []

story.append(Spacer(1, 1.4 * inch))
story.append(pdf_paragraph(title, "PaperTitleCustom"))
story.append(Spacer(1, 0.3 * inch))
story.append(pdf_paragraph("Research Paper", "PaperSubHeading"))
story.append(Spacer(1, 0.25 * inch))
story.append(pdf_paragraph(
    f"Project: Analysis in Global Supply Chain Operations<br/>"
    f"Eligible shipment records analyzed: {total_eligible:,}<br/>"
    f"Generated: {datetime.now().strftime('%d %B %Y')}",
    "PaperBody"
))
story.append(PageBreak())

def pdf_section(story, heading, body):
    story.append(pdf_paragraph(heading, "PaperHeading"))
    pdf_add_text(story, body)

pdf_section(story, "Abstract", abstract)
pdf_section(story, "Introduction", introduction)
pdf_section(story, "Background and Context", background)
pdf_section(story, "Problem Statement", problem_statement)

story.append(pdf_paragraph("Project Objectives", "PaperHeading"))
for item in objectives:
    story.append(pdf_paragraph("- " + item, "PaperBullet"))

pdf_section(story, "Dataset Description", dataset_description)
pdf_add_table(story, dataset_summary_table, "Dataset Summary")

story.append(pdf_paragraph("Dataset Limitations", "PaperHeading"))
for item in limitations:
    story.append(pdf_paragraph("- " + item, "PaperBullet"))

pdf_section(story, "Data Cleaning Methodology", cleaning_methodology)
pdf_section(story, "Delivery Gap Calculation Methodology", gap_methodology)
pdf_section(story, "Delivery Performance Analysis", delivery_analysis)
pdf_add_table(story, classification_table, "Delivery Classification Results")

pdf_section(story, "Shipping Mode Efficiency Analysis", shipping_analysis)
pdf_add_table(story, mode_table, "Shipping Mode Performance")

pdf_section(story, "Regional and Market Analysis", regional_analysis)
pdf_add_table(story, region_table, "Regional Performance")
pdf_add_table(story, market_table, "Market Performance")

if not country_table.empty:
    pdf_add_table(
        story, country_table.head(15),
        "Country Performance - Highest Delay Percentages"
    )

pdf_section(story, "Customer Segment Impact Analysis", segment_analysis)
pdf_add_table(story, segment_table, "Customer Segment Performance")

pdf_section(story, "KPI Methodology", kpi_methodology)
pdf_add_table(story, kpi_table, "Five Required Logistics KPIs")

story.append(pdf_paragraph("Results and Findings", "PaperHeading"))
for item in findings:
    story.append(pdf_paragraph("- " + item, "PaperBullet"))

story.append(pdf_paragraph("Operational Bottlenecks", "PaperHeading"))
for item in bottlenecks:
    story.append(pdf_paragraph("- " + item, "PaperBullet"))

story.append(pdf_paragraph("Business Recommendations", "PaperHeading"))
for index, item in enumerate(recommendations, start=1):
    story.append(pdf_paragraph(f"{index}. {item}", "PaperBullet"))

pdf_section(story, "Conclusion", conclusion)

story.append(pdf_paragraph("References", "PaperHeading"))
for index, item in enumerate(references, start=1):
    story.append(pdf_paragraph(f"{index}. {item}", "PaperBullet"))

pdf_document = SimpleDocTemplate(
    str(PDF_FILE),
    pagesize=A4,
    rightMargin=0.58 * inch,
    leftMargin=0.58 * inch,
    topMargin=0.62 * inch,
    bottomMargin=0.62 * inch,
    title=title,
    author="Global Supply Chain Operations Project"
)

pdf_document.build(
    story,
    onFirstPage=add_page_number,
    onLaterPages=add_page_number
)

print(f"PDF created: {PDF_FILE}")
print(f"PDF size: {PDF_FILE.stat().st_size:,} bytes")


# VERIFY PAPER CONTENT AND EXPORTED FILES

print("\nVERIFYING RESEARCH PAPER EXPORTS")
print("-" * 95)

required_sections = [
    "Abstract",
    "Introduction",
    "Background and Context",
    "Problem Statement",
    "Project Objectives",
    "Dataset Description",
    "Dataset Limitations",
    "Data Cleaning Methodology",
    "Delivery Gap Calculation Methodology",
    "Delivery Performance Analysis",
    "Shipping Mode Efficiency Analysis",
    "Regional and Market Analysis",
    "Customer Segment Impact Analysis",
    "KPI Methodology",
    "Results and Findings",
    "Operational Bottlenecks",
    "Business Recommendations",
    "Conclusion",
    "References"
]

docx_check = DOCX_FILE.exists() and DOCX_FILE.stat().st_size > 5000
pdf_check = PDF_FILE.exists() and PDF_FILE.stat().st_size > 5000

docx_reopened = False
docx_section_checks = {}

if docx_check:
    check_doc = Document(DOCX_FILE)
    docx_text = "\n".join(
        [p.text for p in check_doc.paragraphs]
        + [
            cell.text
            for table in check_doc.tables
            for row in table.rows
            for cell in row.cells
        ]
    )
    docx_reopened = len(docx_text.strip()) > 1000

    for heading in required_sections:
        docx_section_checks[heading] = heading in docx_text

pdf_text_check = False

if pdf_check:
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(PDF_FILE))
        pdf_text = "\n".join(
            page.extract_text() or "" for page in reader.pages
        )
        pdf_text_check = len(pdf_text.strip()) > 1000
    except Exception:
        pdf_text_check = PDF_FILE.stat().st_size > 5000

verification = {
    "generated_at": datetime.now().isoformat(),
    "project": "Analysis in Global Supply Chain Operations",
    "research_paper_title": title,
    "source_dataset": str(GAP_FILE),
    "cleaned_dataset": str(CLEANED_FILE),
    "eligible_shipments_analyzed": int(total_eligible),
    "required_sections": required_sections,
    "docx_file": str(DOCX_FILE),
    "docx_exists": DOCX_FILE.exists(),
    "docx_size_bytes": DOCX_FILE.stat().st_size if DOCX_FILE.exists() else 0,
    "docx_reopened_successfully": docx_reopened,
    "docx_section_checks": docx_section_checks,
    "pdf_file": str(PDF_FILE),
    "pdf_exists": PDF_FILE.exists(),
    "pdf_size_bytes": PDF_FILE.stat().st_size if PDF_FILE.exists() else 0,
    "pdf_text_content_verified": pdf_text_check,
    "delivery_classification_reconciles": (
        delayed_count + exact_on_time_count + early_count == total_eligible
    ),
    "sla_compliance_reconciles": (
        sla_compliant_count == exact_on_time_count + early_count
    ),
    "synthetic_dates_created": False,
    "status": "PASS" if (
        docx_check
        and pdf_check
        and docx_reopened
        and pdf_text_check
        and all(docx_section_checks.values())
        and delayed_count + exact_on_time_count + early_count == total_eligible
        and sla_compliant_count == exact_on_time_count + early_count
    ) else "REVIEW REQUIRED"
}

with open(VERIFICATION_FILE, "w", encoding="utf-8") as file:
    json.dump(verification, file, indent=4, default=str)

print(f"DOCX exists: {'PASS' if verification['docx_exists'] else 'FAIL'}")
print(f"DOCX reopened: {'PASS' if docx_reopened else 'FAIL'}")
print(f"PDF exists: {'PASS' if verification['pdf_exists'] else 'FAIL'}")
print(f"PDF text verification: {'PASS' if pdf_text_check else 'FAIL'}")

for heading, passed in docx_section_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {heading}")

print("\n" + "=" * 95)
print("RESEARCH PAPER DEVELOPMENT SUMMARY")
print("=" * 95)

print(f"Research paper DOCX: {DOCX_FILE}")
print(f"Research paper PDF: {PDF_FILE}")
print(f"Verification report: {VERIFICATION_FILE}")

print(f"DOCX size: {DOCX_FILE.stat().st_size:,} bytes")
print(f"PDF size: {PDF_FILE.stat().st_size:,} bytes")

print(f"\nEligible shipments analyzed: {total_eligible:,}")
print(f"Delayed shipments: {delayed_count:,}")
print(f"Exactly on-time shipments: {exact_on_time_count:,}")
print(f"Early shipments: {early_count:,}")
print(f"SLA compliance: {sla_compliance_pct:.2f}%")
print(f"Average delivery delay: {average_delay:.2f} days")

if verification["status"] == "PASS":
    print("\nRESEARCH PAPER DEVELOPMENT COMPLETED SUCCESSFULLY")
    print("DOCX AND PDF EXPORTS VERIFIED")
else:
    print("\nRESEARCH PAPER EXPORT VERIFICATION REQUIRES REVIEW")

print("=" * 95)

RESEARCH PAPER DEVELOPMENT AND EXPORT
Project directory: C:\Users\hp\Analysis in Global Supply Chain Operations
Cleaned dataset: C:\Users\hp\Analysis in Global Supply Chain Operations\data\processed\cleaned_logistics_data.csv
Delivery gap dataset: C:\Users\hp\Analysis in Global Supply Chain Operations\data\processed\delivery_gap_analysis.csv

LOADING PROJECT DATA
-----------------------------------------------------------------------------------------------
Cleaned dataset shape: (180519, 32)
Delivery gap dataset shape: (180519, 39)
Source analysis records: 180,519
Cleaned dataset records: 180,519
Eligible shipments: 172,765

CALCULATING DELIVERY PERFORMANCE
-----------------------------------------------------------------------------------------------
Delayed shipments: 98,977 (57.29%)
Exactly on-time shipments: 32,196 (18.64%)
Early shipments: 41,592 (24.07%)
SLA-compliant shipments: 73,788 (42.71%)
Average delay among delayed shipments: 1.62 days

CALCULATING SHIPPING MODE PERFORMAN